# Banking footprint analysis

Eighth package. Source: `BCBData.Accounts` (Banco de Portugal account map): relationship
depth and age, account-type mix, institution spread, recent openings.

Design notes:
1. The map lists only OPEN accounts (0 CloseDate in 6,933 records): closures/churn are
   not observable, only openings.
2. Type/mix counts cover "Titular" accounts; authorized-to-move and beneficial-owner
   roles are reported separately (access vs ownership).
3. Card accounts (`SubType = Cartão`) are a subset of credit facilities
   (`AccountType = Abertura de crédito`): the counts overlap by design.
4. 203 credit-facility records have no institution identification (empty code and
   name): they count in type totals but not in `n_institutions`.

In [ ]:
import json
import sys

import pandas as pd

sys.path.insert(0, ".")
from banking_features import banking_features, render_profile_text

pd.set_option("display.width", 200)
raw = pd.read_csv("../data/data_full.csv")
snap = raw.drop_duplicates("uuid")[["uuid", "value"]].reset_index(drop=True)
feats = pd.DataFrame([
    {"uuid": u, **banking_features(json.loads(v))}
    for u, v in zip(snap["uuid"], snap["value"])
])
print(f"{feats.shape[0]} proponents x {feats.shape[1] - 1} fields, "
      f"version {feats['features_version'].iloc[0]}")

## 1. Population overview

In [ ]:
print("with own (Titular) accounts:", int((feats["n_as_titular"].fillna(0) > 0).sum()),
      "of", len(feats))
KEY = ["n_as_titular", "n_institutions", "oldest_account_years",
       "n_credit_facilities", "n_current_accounts", "n_opened_12m",
       "n_credit_opened_12m", "main_institution_share"]
print(feats[KEY].describe(percentiles=[.25, .5, .75, .9]).round(2).T.to_string())
print("\nmarkers: savings", int((feats["has_savings"] == 1).sum()),
      "| investments", int((feats["has_investments"] == 1).sum()),
      "| single-bank", int((feats["main_institution_share"] >= 0.999).sum()),
      "| credit opening in last 12m", int((feats["n_credit_opened_12m"].fillna(0) > 0).sum()))

## 2. Layer 2 examples: deep-rooted, scattered, and no-data

In [ ]:
fset = feats.set_index("uuid")
vmap = dict(zip(snap["uuid"], snap["value"]))
examples = [
    fset[(fset["oldest_account_years"] > 20) & (fset["has_investments"] == 1)].index[0],
    fset[(fset["n_institutions"] >= 7)].index[0],
    fset[fset["has_bcb_data"] == 0].index[0],
]
for u in examples:
    print(f"=== {u[:8]} ===")
    print(render_profile_text(banking_features(json.loads(vmap[u]))))
    print()

## 3. Validation: module vs independent pandas recomputation. Must print 0.

In [ ]:
def dig(d, *path):
    for k in path:
        d = d.get(k) if isinstance(d, dict) else None
    return d

mism = checked = 0
for uuid, v in zip(snap["uuid"], snap["value"]):
    p = json.loads(v)
    vec = banking_features(p)
    A = pd.DataFrame(dig(p, "BCBData", "Accounts") or [])
    if A.empty:
        continue
    own = A[A["AccountRelation"] == "Titular"]
    if own.empty:
        continue
    codes = own["InstitutionCode"].fillna("")
    expect = {
        "n_as_titular": len(own),
        "n_credit_facilities": int((own["AccountType"] == "Abertura de crédito").sum()),
        "n_current_accounts": int((own["SubType"] == "Depósito à ordem").sum()),
        "n_term_deposits": int(own["SubType"].isin(
            ["Depósito a prazo", "Depósito com pré-aviso"]).sum()),
        "n_investment_accounts": int((own["AccountType"] == "Instrumentos financeiros").sum()),
        "n_institutions": codes[codes != ""].nunique(),
    }
    doc = dig(p, "BCBData", "DocumentDate")
    if doc:
        ref = int(doc[:4]) * 12 + int(doc[5:7]) - 1
        om = own["OpenDate"].dropna().map(lambda s: int(s[:4]) * 12 + int(s[5:7]) - 1)
        if len(om):
            expect["oldest_account_years"] = round((ref - om.min()) / 12, 1)
            expect["n_opened_12m"] = int((ref - om < 12).sum())
    for k, e in expect.items():
        checked += 1
        if vec[k] != e:
            mism += 1
            print("MISMATCH", uuid[:8], k, vec[k], e)
print(f"{checked} comparisons, mismatches: {mism}")